# Adobe MLOps Project — Exploratory Analysis & Experimentation

Use this notebook for:
- Exploring your dataset
- Running quick model experiments
- Visualising drift results


In [ ]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import models

mlflow.set_tracking_uri('http://localhost:5000')
print('MLflow connected:', mlflow.get_tracking_uri())

## 1. List MLflow experiments

In [ ]:
client = mlflow.tracking.MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    print(f'{exp.experiment_id}: {exp.name}')

## 2. Compare runs

In [ ]:
runs = mlflow.search_runs(experiment_names=['image_classifier'])
if not runs.empty:
    cols = ['run_id','metrics.best_val_acc','params.epochs','params.lr','tags.mlflow.runName']
    available = [c for c in cols if c in runs.columns]
    print(runs[available].sort_values('metrics.best_val_acc', ascending=False).head(10))
else:
    print('No runs yet — run model/train.py first.')

## 3. Run drift detection manually

In [ ]:
import sys
sys.path.insert(0, '..')
from monitoring.drift_detector import load_reference_data, load_current_data, run_drift_analysis

ref  = load_reference_data()
curr = load_current_data()
results = run_drift_analysis(ref, curr)

print('=== Drift Results ===')
for k, v in results.items():
    if k != 'report_path':
        print(f'  {k}: {v}')
print(f"\nHTML report: {results.get('report_path')}")

## 4. Visualise feature drift

In [ ]:
feature_cols = [c for c in ref.columns if c.startswith('feature_')]

fig, axes = plt.subplots(1, len(feature_cols), figsize=(4*len(feature_cols), 4))
for ax, col in zip(axes, feature_cols):
    ax.hist(ref[col],  bins=30, alpha=0.6, label='Reference', color='steelblue')
    ax.hist(curr[col], bins=30, alpha=0.6, label='Current',   color='coral')
    ax.set_title(col.replace('feature_','').replace('_',' '))
    ax.legend(fontsize=8)

plt.suptitle('Feature Distribution: Reference vs Current', y=1.02)
plt.tight_layout()
plt.show()

## 5. Test A/B routing logic

In [ ]:
from app.model_manager import ModelManager

mm = ModelManager()
mm.ab_config = {'stable': 0.8, 'canary': 0.2}

import random, string
results = [mm.route_ab('u_' + ''.join(random.choices(string.ascii_lowercase, k=6)))
           for _ in range(1000)]

stable_pct = results.count('stable') / len(results)
canary_pct = results.count('canary') / len(results)
print(f'Stable: {stable_pct:.1%} | Canary: {canary_pct:.1%}')

plt.bar(['stable', 'canary'], [stable_pct, canary_pct], color=['steelblue','coral'])
plt.title('A/B Traffic Split (1000 users)')
plt.ylabel('Fraction')
plt.ylim(0, 1)
plt.show()